# Dataset (v2) — processed_v2 için birleşik veri hazırlama
Bu notebook `dataset.ipynb`'deki temel akışı alır ve `processed_v2`'deki `Bolge_*` one-hot sütunlarını otomatik olarak destekleyecek şekilde veri yükleme, ölçekleme ve bölme işlemlerini yapar.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import joblib

In [ ]:
# Ham veriyi oku ve preprocessing.py ile aynı adımları uygula
from pathlib import Path
raw_path = Path('../../data/raw/hamdata.csv')
print('Reading raw data from', raw_path)

df_raw = pd.read_csv(raw_path, encoding='utf-8', low_memory=False)
print(f'Total rows (raw): {len(df_raw)}')

# Tarih parse ve cyclical encoding
df_raw['Date'] = pd.to_datetime(df_raw['Date'], format='%d/%m/%Y %H:%M:%S', errors='coerce')
df_raw = df_raw.dropna(subset=['Date'])

df_raw['Year'] = df_raw['Date'].dt.year
df_raw['Month'] = df_raw['Date'].dt.month
df_raw['Day'] = df_raw['Date'].dt.day

df_raw['Month_sin'] = np.sin(2 * np.pi * df_raw['Month'] / 12)
df_raw['Month_cos'] = np.cos(2 * np.pi * df_raw['Month'] / 12)
df_raw['Day_sin'] = np.sin(2 * np.pi * df_raw['Day'] / 31)
df_raw['Day_cos'] = np.cos(2 * np.pi * df_raw['Day'] / 31)

# Depth filtrele (<=150)
initial_count = len(df_raw)
df_raw = df_raw[df_raw['Depth'] <= 150].copy()
print(f'Depth > 150 removed: {initial_count - len(df_raw)} rows')

# Magnitude kategorisi
def risk_siniflandir(buyukluk):
    if buyukluk < 4.5:
        return 'Hafif'
    elif buyukluk < 6.0:
        return 'Orta'
    else:
        return 'Yıkıcı'

df_raw['Magnitude_Category'] = df_raw['Magnitude'].apply(risk_siniflandir)

# Sadece gerekli sütunları al
keep_cols = ['Year', 'Month_sin', 'Month_cos', 'Day_sin', 'Day_cos',
             'Latitude', 'Longitude', 'Depth', 'Magnitude', 'Magnitude_Category']

# Eğer ham veri bu sütunlardan eksikse, hata verme; mevcut olanları kullan
missing = [c for c in keep_cols if c not in df_raw.columns]
if missing:
    raise KeyError(f"Expected columns missing in raw data: {missing}")

df = df_raw[keep_cols].copy()
print('Processed df shape:', df.shape)
print('Columns:', list(df.columns))

In [ ]:
# Hedef sütun ve girdi sütunlarını otomatik belirle
target_col = 'Magnitude'
input_cols = [c for c in df.columns if c != target_col]
# Sürekli (scale edilecek) sütunlar: sayısal ve ikiden fazla benzersiz değere sahip olanlar
continuous_cols = [
    c for c in input_cols
    if pd.api.types.is_numeric_dtype(df[c]) and df[c].nunique() > 2
]
# Binary / one-hot gibi gözüken sütunları ayrı tutuyoruz (örn. Bolge_*)
binary_cols = [c for c in input_cols if pd.api.types.is_numeric_dtype(df[c]) and df[c].nunique() <= 2]
other_cols = [c for c in input_cols if c not in continuous_cols + binary_cols]

print('Detected input columns count:', len(input_cols))
print('Continuous (will scale):', continuous_cols)
print('Binary/one-hot (kept as-is):', binary_cols)
print('Other columns (kept as-is):', other_cols)

In [ ]:
# preprocessing.py ile uyumlu split ve scaler uygulaması
from pathlib import Path
out_dir = Path('data/processed_v2')
out_dir.mkdir(parents=True, exist_ok=True)

# Özellik ve hedef
X = df.drop(columns=['Magnitude', 'Magnitude_Category'])
y = df['Magnitude']

# Train/test split (preprocessing.py ile aynı)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Train rows: {len(X_train)}, Test rows: {len(X_test)}")

# Scaler: fit yalnızca X_train'e
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test.columns)

# Kaydet (processed_v2 içine)
X_train_scaled_df.to_csv(out_dir / 'X_train.csv', index=False)
pd.DataFrame(y_train, columns=['Magnitude']).to_csv(out_dir / 'y_train.csv', index=False)
X_test_scaled_df.to_csv(out_dir / 'X_test.csv', index=False)
pd.DataFrame(y_test, columns=['Magnitude']).to_csv(out_dir / 'y_test.csv', index=False)

# Kategori sütunlarını orijinal indeksle kaydet (preprocessing.py ile aynı)
y_train_cat = df.loc[X_train.index, 'Magnitude_Category']
y_test_cat = df.loc[X_test.index, 'Magnitude_Category']
pd.DataFrame(y_train_cat).to_csv(out_dir / 'y_train_category.csv', index=False)
pd.DataFrame(y_test_cat).to_csv(out_dir / 'y_test_category.csv', index=False)

# Scaler ve feature listesi
joblib.dump(scaler, out_dir / 'deprem_scaler_v2.pkl')
with open(out_dir / 'scaler_features.txt', 'w', encoding='utf-8') as f:
    f.write(','.join(X_train.columns.tolist()))

print('[SAVED] Files written to', out_dir)
print('X_train shape:', X_train_scaled_df.shape)
print('X_test shape:', X_test_scaled_df.shape)